# BioNNE-L: Dense Retriever Fine-Tuning

This notebook fine-tunes a dense retriever on `mention -> gold concept` pairs with an in-batch contrastive objective, then evaluates the resulting model with the dense-only retrieval pipeline.


In [ ]:
import copy
import gc
import json
import logging
import os
import random
from pathlib import Path

import mlflow
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from lib.data.text_preprocessing import preprocess_text
from lib.data.vocab_enrichment import (
    enrich_vocab_with_oov_train_dev_terms,
    filter_vocab_for_dataset_language,
    prepare_experiment_vocab,
)
from lib.retrieval.pipelines import evaluate_dense_retrieval, predict_dense_to_path
from lib.retrieval.retriever_training.dense import train_dense_retriever
from lib.retrieval.retriever_training.data import build_dense_training_pairs
from lib.retrieval.tuning import evaluate_dev_predictions
from lib.utils.logging_utils import configure_logging


In [ ]:
configure_logging(level=logging.INFO, force=True)

ARTIFACTS_DIR = "./artifacts_dense_finetuning"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

MLFLOW_EXPERIMENT_PREFIX = "bionnel"

DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_DEVICE


In [ ]:
LOCAL_MLRUNS_DIR = Path("mlruns").resolve()
mlflow.set_tracking_uri(LOCAL_MLRUNS_DIR.as_uri())
print("MLflow tracking URI:", mlflow.get_tracking_uri())


In [ ]:
## Common Hyperparameters
COMMON_TRAINING_CONFIG = {
    "SEED": 42,  # Random seed for training, splitting, and hard-negative mining.
    "NUM_HARD_NEGATIVES": 0,  # Offline hard negatives mined per positive pair; 0 disables mining.
    "HARD_NEGATIVE_DEDUPLICATE_BY_CUI": True,  # Deduplicate mined hard negatives by CUI.
    "HARD_NEGATIVE_SKIP_TOPK": 0,  # Skip this many strongest non-gold candidates before taking hard negatives.
    "HARD_NEGATIVE_CACHE_DIR": "artifacts_dense_finetuning/cache/hard_negatives",  # Local cache for mined hard negatives.
    "EVAL_BATCH_SIZE": 128,  # Evaluation batch size.
    "EPOCHS": 10,  # Number of fine-tuning epochs.
    "TRAIN_BATCH_SIZE": 64,  # Per-device training batch size.
    "CUI_AWARE_BATCHING": True,  # Avoid duplicate positive CUIs inside in-batch negative batches.
    "BATCH_DEDUPLICATE_BY_TEXT": False,  # Also avoid duplicate mention/concept texts inside a batch.
    "LEARNING_RATE": 2e-5,  # Optimizer learning rate.
    "WEIGHT_DECAY": 0.01,  # Optimizer weight decay.
    "WARMUP_RATIO": 0.1,  # Warmup fraction for the learning-rate schedule.
    "USE_FP16": False,  # Enable fp16 mixed precision.
    "USE_BF16": False,  # Enable bf16 mixed precision.
    "ALLOW_TF32": False,  # Allow TF32 CUDA kernels.
    "MAX_SEQ_LENGTH": 64,  # Maximum token length for dense retriever inputs.
    "GRAD_ACCUMULATION_STEPS": 1,  # Gradient accumulation steps.
    "TRAIN_LOGGING_STEPS": 50,  # Training log interval in optimizer steps.
    "DEV_LOSS_EVAL_STEPS": 0,  # 0 evaluates loss by epoch; positive values evaluate by steps.
    "EVAL_EVERY_EPOCH": True,  # Run retrieval-quality model selection after each epoch.
    "EARLY_STOPPING_PATIENCE": 3,  # Stop after this many non-improving epochs.
    "SELECTION_METRIC": "Acc@20",  # Dev metric used to select the best checkpoint.
    "RUN_NAME": "ru-dense-finetuning-infonce-no-hard-negatives",  # Optional run name for logging.
}

COMMON_INFERENCE_CONFIG = {
    "ENRICH_VOCABULARY": False,  # Optionally add all unique train/dev mention-CUI pairs beyond the default OOV-only step.
    "TEST_ENRICH_VOCABULARY": True,  # Apply the same optional train/dev-only enrichment during test inference; never use test mentions.
    "DEDUPLICATE_BY_CUI": True,  # Keep at most one candidate per CUI in top-k results.
    "DEV_TOPK": 20,  # Number of dev predictions retained for evaluation.
    "TEST_TOPK": 5,  # Number of final test predictions written per mention.
    "QUERY_BATCH_SIZE": 131_072,  # Number of mentions processed per retrieval batch.
    "DENSE_VOCAB_BATCH_SIZE": 16_384,  # Vocabulary chunk size for dense scoring.
    "ST_ENCODE_BATCH_SIZE": 8_192,  # SentenceTransformer encoding batch size.
}


## Data Loading


In [ ]:
ru_data_train = pd.read_parquet("data/parquet/ru/bionnel_ru_train.parquet")
ru_data_dev = pd.read_parquet("data/parquet/ru/bionnel_ru_dev.parquet")
ru_data_test = pd.read_csv("data/tsv/ru/bionnel_ru_test.tsv", sep="	")

en_data_train = pd.read_parquet("data/parquet/en/bionnel_en_train.parquet")
en_data_dev = pd.read_parquet("data/parquet/en/bionnel_en_dev.parquet")
en_data_test = pd.read_csv("data/tsv/en/bionnel_en_test.tsv", sep="	")

bilingual_data_train = pd.read_parquet("data/parquet/bilingual/bionnel_bilingual_train.parquet")
bilingual_data_dev = pd.read_parquet("data/parquet/bilingual/bionnel_bilingual_dev.parquet")
bilingual_data_test = pd.read_csv("data/tsv/bilingual/bionnel_bilingual_test.tsv", sep="	")

vocab = pd.read_parquet("data/vocabular/bionnel_vocab_bilingual.parquet")

for dataset_df in [
    ru_data_train,
    ru_data_dev,
    ru_data_test,
    en_data_train,
    en_data_dev,
    en_data_test,
    bilingual_data_train,
    bilingual_data_dev,
    bilingual_data_test,
]:
    dataset_df["text"] = dataset_df["text"].map(preprocess_text)

vocab["concept_name"] = vocab["concept_name"].map(preprocess_text)

RU_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat(
    [
        ru_data_train,
        ru_data_dev,
        en_data_train,
        en_data_dev,
        bilingual_data_train,
        bilingual_data_dev,
    ],
    ignore_index=True,
)
EN_ENTITIES_FOR_VOCAB_ENRICHMENT = pd.concat(
    [
        en_data_train,
        en_data_dev,
    ],
    ignore_index=True,
)
BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT = RU_ENTITIES_FOR_VOCAB_ENRICHMENT.copy()

print("RU train/dev/test:", ru_data_train.shape, ru_data_dev.shape, ru_data_test.shape)
print("EN train/dev/test:", en_data_train.shape, en_data_dev.shape, en_data_test.shape)
print("Bilingual train/dev/test:", bilingual_data_train.shape, bilingual_data_dev.shape, bilingual_data_test.shape)
print("Base vocabulary shape:", vocab.shape)
print("RU vocab enrichment pool:", RU_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print("EN-only vocab enrichment pool:", EN_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print("Bilingual vocab enrichment pool:", BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT.shape)
print("Semantic types:", sorted(vocab["semantic_type"].dropna().unique().tolist()))


## Training Utils


In [ ]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_artifact_dir(dataset_name):
    artifact_dir = Path(ARTIFACTS_DIR) / dataset_name.lower()
    artifact_dir.mkdir(parents=True, exist_ok=True)
    return artifact_dir


def build_base_experiment_config(model_name):
    return {
        "MODEL_NAME": model_name,
        "DEVICE": DEFAULT_DEVICE,
        "TRAINING": copy.deepcopy(COMMON_TRAINING_CONFIG),
        "INFERENCE": copy.deepcopy(COMMON_INFERENCE_CONFIG),
    }


def build_runtime_config(cfg):
    runtime_cfg = {
        "MODEL_NAME": cfg["MODEL_NAME"],
        "DEVICE": cfg["DEVICE"],
    }
    runtime_cfg.update(copy.deepcopy(cfg["TRAINING"]))
    runtime_cfg.update(copy.deepcopy(cfg["INFERENCE"]))
    runtime_cfg["TRAINING"] = copy.deepcopy(cfg["TRAINING"])
    runtime_cfg["INFERENCE"] = copy.deepcopy(cfg["INFERENCE"])
    return runtime_cfg




def build_test_inference_config(cfg):
    test_cfg = copy.deepcopy(cfg)
    test_cfg["INFERENCE"]["ENRICH_VOCABULARY"] = bool(
        cfg["INFERENCE"].get("TEST_ENRICH_VOCABULARY", cfg["INFERENCE"].get("ENRICH_VOCABULARY", False))
    )
    return test_cfg


def build_test_inference_vocab(base_vocab_df, enrichment_entities_df, cfg, *, dataset_name=None, lang_value=None):
    test_cfg = build_test_inference_config(cfg)
    test_vocab_df = enrich_vocab_with_oov_train_dev_terms(
        base_vocab_df.copy(),
        enrichment_entities_df,
        lang_value=lang_value,
    )
    test_vocab_df = prepare_experiment_vocab(
        test_vocab_df,
        enrichment_entities_df,
        test_cfg["INFERENCE"],
        lang_value=lang_value,
    )
    if dataset_name is not None:
        test_vocab_df = filter_vocab_for_dataset_language(test_vocab_df, dataset_name)
    return test_vocab_df

def build_effective_experiment_config(base_cfg, training_result):
    effective_cfg = copy.deepcopy(base_cfg)
    effective_cfg["TRAINING"].update({
        "BEST_EPOCH": int(training_result["best_epoch"]),
        "BEST_MODEL_DIR": str(training_result["best_model_dir"]),
        "BEST_CHECKPOINT_DIR": str(training_result["best_checkpoint_dir"]),
        "NUM_TRAIN_PAIRS": int(len(training_result["train_pairs_df"])),
    })
    return effective_cfg


def run_training(dataset_name, train_df, dev_df, vocab_df, cfg):
    artifact_dir = get_artifact_dir(dataset_name)
    train_output_dir = artifact_dir / "training"
    runtime_cfg = build_runtime_config(cfg)
    set_seed(runtime_cfg["SEED"])
    training_result = train_dense_retriever(
        train_df=train_df,
        dev_df=dev_df,
        vocab_df=vocab_df,
        model_name=runtime_cfg["MODEL_NAME"],
        output_dir=train_output_dir,
        cfg=runtime_cfg,
    )
    effective_cfg = build_effective_experiment_config(cfg, training_result)
    return training_result, effective_cfg


## Prediction Utils


In [ ]:
def load_best_model(effective_cfg):
    return SentenceTransformer(
        effective_cfg["TRAINING"]["BEST_MODEL_DIR"],
        device=effective_cfg["DEVICE"],
    )


def evaluate_on_dev(data_df, vocab_df, st_model, cfg, resource_cache=None):
    if resource_cache is None:
        resource_cache = {}
    runtime_cfg = build_runtime_config(cfg)
    return evaluate_dense_retrieval(
        data_df=data_df,
        vocab_df=vocab_df,
        st_model=st_model,
        topk=runtime_cfg["DEV_TOPK"],
        query_batch_size=runtime_cfg["QUERY_BATCH_SIZE"],
        dense_vocab_batch_size=runtime_cfg["DENSE_VOCAB_BATCH_SIZE"],
        st_encode_batch_size=runtime_cfg["ST_ENCODE_BATCH_SIZE"],
        deduplicate_by_cui=runtime_cfg["DEDUPLICATE_BY_CUI"],
        resource_cache=resource_cache,
    )


def predict_on_test(data_df, vocab_df, st_model, cfg, output_path, resource_cache=None):
    if resource_cache is None:
        resource_cache = {}
    runtime_cfg = build_runtime_config(cfg)
    return predict_dense_to_path(
        data_df=data_df,
        vocab_df=vocab_df,
        st_model=st_model,
        output_path=output_path,
        topk=runtime_cfg["TEST_TOPK"],
        query_batch_size=runtime_cfg["QUERY_BATCH_SIZE"],
        dense_vocab_batch_size=runtime_cfg["DENSE_VOCAB_BATCH_SIZE"],
        st_encode_batch_size=runtime_cfg["ST_ENCODE_BATCH_SIZE"],
        deduplicate_by_cui=runtime_cfg["DEDUPLICATE_BY_CUI"],
        resource_cache=resource_cache,
    )


## MLflow Utils


In [ ]:
def flatten_config_for_mlflow(prefix, value):
    if isinstance(value, dict):
        flat = {}
        for key, nested_value in value.items():
            nested_prefix = f"{prefix}.{key}" if prefix else str(key)
            flat.update(flatten_config_for_mlflow(nested_prefix, nested_value))
        return flat
    if isinstance(value, (list, tuple)):
        return {prefix: str(list(value))}
    return {prefix: value}


def build_mlflow_params(dataset_name, base_cfg, effective_cfg):
    training_cfg = effective_cfg["TRAINING"]
    inference_cfg = effective_cfg["INFERENCE"]
    return {
        "dataset_name": dataset_name,
        "model_name": effective_cfg["MODEL_NAME"],
        "device": effective_cfg["DEVICE"],
        "seed": int(training_cfg["SEED"]),
        "epochs": int(training_cfg["EPOCHS"]),
        "train_batch_size": int(training_cfg["TRAIN_BATCH_SIZE"]),
        "eval_batch_size": int(training_cfg.get("EVAL_BATCH_SIZE", training_cfg["TRAIN_BATCH_SIZE"])),
        "cui_aware_batching": bool(training_cfg.get("CUI_AWARE_BATCHING", False)),
        "batch_deduplicate_by_text": bool(training_cfg.get("BATCH_DEDUPLICATE_BY_TEXT", False)),
        "train_logging_steps": int(training_cfg.get("TRAIN_LOGGING_STEPS", 50)),
        "dev_loss_eval_steps": int(training_cfg.get("DEV_LOSS_EVAL_STEPS", 0)),
        "learning_rate": float(training_cfg["LEARNING_RATE"]),
        "weight_decay": float(training_cfg["WEIGHT_DECAY"]),
        "warmup_ratio": float(training_cfg["WARMUP_RATIO"]),
        "max_seq_length": int(training_cfg["MAX_SEQ_LENGTH"]),
        "use_fp16": bool(training_cfg.get("USE_FP16", False)),
        "use_bf16": bool(training_cfg.get("USE_BF16", False)),
        "allow_tf32": bool(training_cfg.get("ALLOW_TF32", False)),
        "grad_accumulation_steps": int(training_cfg["GRAD_ACCUMULATION_STEPS"]),
        "early_stopping_patience": int(training_cfg.get("EARLY_STOPPING_PATIENCE", 3)),
        "selection_metric": training_cfg["SELECTION_METRIC"],
        "best_epoch": int(training_cfg["BEST_EPOCH"]),
        "num_train_pairs": int(training_cfg["NUM_TRAIN_PAIRS"]),
        "num_hard_negatives": int(training_cfg.get("NUM_HARD_NEGATIVES", 0)),
        "hard_negative_deduplicate_by_cui": bool(training_cfg.get("HARD_NEGATIVE_DEDUPLICATE_BY_CUI", True)),
        "hard_negative_skip_topk": int(training_cfg.get("HARD_NEGATIVE_SKIP_TOPK", 0)),
        "hard_negative_cache_dir": training_cfg.get("HARD_NEGATIVE_CACHE_DIR"),
        "enrich_vocabulary": bool(inference_cfg.get("ENRICH_VOCABULARY", False)),
        "deduplicate_by_cui": bool(inference_cfg["DEDUPLICATE_BY_CUI"]),
        "dev_topk": int(inference_cfg["DEV_TOPK"]),
        "test_topk": int(inference_cfg["TEST_TOPK"]),
        "query_batch_size": int(inference_cfg["QUERY_BATCH_SIZE"]),
        "dense_vocab_batch_size": int(inference_cfg["DENSE_VOCAB_BATCH_SIZE"]),
        "st_encode_batch_size": int(inference_cfg["ST_ENCODE_BATCH_SIZE"]),
        "test_enrich_vocabulary": bool(inference_cfg.get("TEST_ENRICH_VOCABULARY", inference_cfg.get("ENRICH_VOCABULARY", False))),
    }


def log_run_to_mlflow(dataset_name, base_cfg, effective_cfg, dev_metrics, test_metrics, artifact_paths, training_result):
    experiment_name = f"{MLFLOW_EXPERIMENT_PREFIX}-{dataset_name.lower()}"
    mlflow.set_experiment(experiment_name)
    run_name = effective_cfg["TRAINING"].get("RUN_NAME") or f"{dataset_name.lower()}-dense-finetuning"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(build_mlflow_params(dataset_name, base_cfg, effective_cfg))
        mlflow_metrics = {}
        for metric_name, metric_value in dev_metrics.items():
            normalized_name = metric_name.lower().replace("@", "_at_")
            mlflow_metrics[f"dev_{normalized_name}"] = float(metric_value)
        for metric_name, metric_value in test_metrics.items():
            normalized_name = metric_name.lower().replace("@", "_at_")
            mlflow_metrics[f"test_{normalized_name}"] = float(metric_value)
        mlflow.log_metrics(mlflow_metrics)

        history_df = training_result["history_df"].copy()
        if not history_df.empty:
            if "step" not in history_df.columns:
                history_df["step"] = range(1, len(history_df) + 1)
            dev_acc_history_columns = [
                column_name
                for column_name in history_df.columns
                if isinstance(column_name, str) and column_name.startswith("dev_Acc@")
            ]
            for _, row in history_df.iterrows():
                step = int(row["step"]) if "step" in row and row["step"] == row["step"] else None
                if "loss" in row and row["loss"] == row["loss"]:
                    mlflow.log_metric("train_loss", float(row["loss"]), step=step)
                if "eval_loss" in row and row["eval_loss"] == row["eval_loss"]:
                    mlflow.log_metric("dev_loss", float(row["eval_loss"]), step=step)
            if dev_acc_history_columns and "epoch_int" in history_df.columns:
                dev_acc_history_df = history_df.dropna(subset=["epoch_int"]).copy()
                dev_acc_history_df = dev_acc_history_df.dropna(how="all", subset=dev_acc_history_columns)
                if not dev_acc_history_df.empty:
                    dev_acc_history_df["epoch_int"] = dev_acc_history_df["epoch_int"].astype(int)
                    dev_acc_history_df = dev_acc_history_df.sort_values(["epoch_int", "step"], kind="stable")
                    dev_acc_history_df = dev_acc_history_df.groupby("epoch_int", as_index=False)[dev_acc_history_columns].last()
                    for _, row in dev_acc_history_df.iterrows():
                        epoch_step = int(row["epoch_int"])
                        for column_name in dev_acc_history_columns:
                            if row[column_name] == row[column_name]:
                                normalized_name = column_name.removeprefix("dev_").lower().replace("@", "_at_")
                                mlflow.log_metric(f"dev_{normalized_name}_history", float(row[column_name]), step=epoch_step)

        artifact_dir_map = {
            "best_model": "model",
            "dev_predictions": "predictions",
            "test_predictions": "predictions",
            "test_metrics": "metrics",
            "metrics_summary": "metrics",
            "base_config": "configs",
            "effective_config": "configs",
            "training_history": "training",
        }
        for artifact_name, artifact_path in artifact_paths.items():
            target_dir = artifact_dir_map.get(artifact_name)
            if target_dir is None:
                continue
            if Path(artifact_path).is_dir():
                mlflow.log_artifacts(artifact_path, artifact_path=target_dir)
            else:
                mlflow.log_artifact(artifact_path, artifact_path=target_dir)

        return mlflow.active_run().info.run_id


## Artifacts Utils


In [ ]:
def save_local_artifacts(
    dataset_name,
    base_cfg,
    effective_cfg,
    dev_predictions_df,
    dev_metrics,
    test_predictions_df,
    test_metrics,
    training_result,
    test_predictions_path=None,
):
    artifact_dir = get_artifact_dir(dataset_name)

    if test_predictions_path is None:
        test_predictions_path = artifact_dir / "test_predictions.tsv"
    else:
        test_predictions_path = Path(test_predictions_path)

    dev_predictions_path = artifact_dir / "dev_predictions.tsv"
    test_metrics_path = artifact_dir / "test_metrics.json"
    metrics_table_path = artifact_dir / "metrics_summary.tsv"
    base_config_path = artifact_dir / "base_config.json"
    effective_config_path = artifact_dir / "effective_config.json"
    training_history_path = artifact_dir / "training_history.tsv"
    training_pairs_path = artifact_dir / "training_pairs.tsv"

    dev_predictions_df.to_csv(dev_predictions_path, sep="	", index=False)
    if not test_predictions_path.exists():
        test_predictions_df.to_csv(test_predictions_path, sep="	", index=False)
    test_metrics_path.write_text(json.dumps(test_metrics, indent=2, ensure_ascii=False))
    pd.DataFrame([
        {"split": "dev", **dev_metrics},
        {"split": "test", **test_metrics},
    ]).to_csv(metrics_table_path, sep="	", index=False)
    base_config_path.write_text(json.dumps(base_cfg, indent=2, ensure_ascii=False))
    effective_config_path.write_text(json.dumps(effective_cfg, indent=2, ensure_ascii=False))
    training_result["history_df"].to_csv(training_history_path, sep="	", index=False)
    training_result["train_pairs_df"].to_csv(training_pairs_path, sep="	", index=False)

    return {
        "best_model": str(effective_cfg["TRAINING"]["BEST_MODEL_DIR"]),
        "dev_predictions": str(dev_predictions_path),
        "test_predictions": str(test_predictions_path),
        "test_metrics": str(test_metrics_path),
        "metrics_summary": str(metrics_table_path),
        "base_config": str(base_config_path),
        "effective_config": str(effective_config_path),
        "training_history": str(training_history_path),
        "training_pairs": str(training_pairs_path),
    }


## RU Experiment


In [ ]:
RU_CONFIG = build_base_experiment_config("andorei/BERGAMOT-multilingual-GAT")

In [ ]:
ru_vocab = enrich_vocab_with_oov_train_dev_terms(vocab.copy(), RU_ENTITIES_FOR_VOCAB_ENRICHMENT)
ru_vocab = prepare_experiment_vocab(ru_vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_CONFIG["INFERENCE"])
ru_training_result, RU_EFFECTIVE_CONFIG = run_training(
    dataset_name="ru",
    train_df=ru_data_train,
    dev_df=ru_data_dev,
    vocab_df=ru_vocab,
    cfg=RU_CONFIG,
)
len(ru_training_result["train_pairs_df"]), RU_EFFECTIVE_CONFIG["TRAINING"]["BEST_EPOCH"], RU_EFFECTIVE_CONFIG["TRAINING"]["NUM_HARD_NEGATIVES"]


In [ ]:
ru_best_st_model = load_best_model(RU_EFFECTIVE_CONFIG)
ru_dev_resource_cache = {}
ru_dev_predictions_df, ru_dev_metrics = evaluate_on_dev(
    data_df=ru_data_dev,
    vocab_df=ru_vocab,
    st_model=ru_best_st_model,
    cfg=RU_EFFECTIVE_CONFIG,
    resource_cache=ru_dev_resource_cache,
)
RU_EFFECTIVE_CONFIG, ru_dev_metrics


In [ ]:
ru_test_vocab = build_test_inference_vocab(vocab, RU_ENTITIES_FOR_VOCAB_ENRICHMENT, RU_EFFECTIVE_CONFIG, dataset_name="ru")
ru_test_resource_cache = {}
ru_test_predictions_df, ru_test_predictions_path = predict_on_test(
    data_df=ru_data_test,
    vocab_df=ru_test_vocab,
    st_model=ru_best_st_model,
    cfg=RU_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir("ru") / "test_predictions.tsv",
    resource_cache=ru_test_resource_cache,
)
ru_test_metrics = evaluate_dev_predictions(predictions_df=ru_test_predictions_df, data_df=ru_data_test)
ru_test_metrics


In [ ]:
ru_artifact_paths = save_local_artifacts(
    dataset_name="ru",
    base_cfg=RU_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    dev_predictions_df=ru_dev_predictions_df,
    dev_metrics=ru_dev_metrics,
    test_predictions_df=ru_test_predictions_df,
    test_metrics=ru_test_metrics,
    training_result=ru_training_result,
    test_predictions_path=ru_test_predictions_path,
)
ru_artifact_paths


In [ ]:
ru_mlflow_run_id = log_run_to_mlflow(
    dataset_name="ru",
    base_cfg=RU_CONFIG,
    effective_cfg=RU_EFFECTIVE_CONFIG,
    dev_metrics=ru_dev_metrics,
    test_metrics=ru_test_metrics,
    training_result=ru_training_result,
    artifact_paths=ru_artifact_paths,
)
ru_artifact_paths, ru_mlflow_run_id


In [ ]:
del ru_best_st_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## EN Experiment


In [ ]:
EN_CONFIG = build_base_experiment_config("andorei/BERGAMOT-multilingual-GAT")

In [ ]:
en_vocab = enrich_vocab_with_oov_train_dev_terms(
    vocab.copy(),
    EN_ENTITIES_FOR_VOCAB_ENRICHMENT,
    lang_value="EN",
)
en_vocab = prepare_experiment_vocab(
    en_vocab,
    EN_ENTITIES_FOR_VOCAB_ENRICHMENT,
    EN_CONFIG["INFERENCE"],
    lang_value="EN",
)
en_vocab = filter_vocab_for_dataset_language(en_vocab, "en")
en_training_result, EN_EFFECTIVE_CONFIG = run_training(
    dataset_name="en",
    train_df=en_data_train,
    dev_df=en_data_dev,
    vocab_df=en_vocab,
    cfg=EN_CONFIG,
)
len(en_training_result["train_pairs_df"]), EN_EFFECTIVE_CONFIG["TRAINING"]["BEST_EPOCH"], EN_EFFECTIVE_CONFIG["TRAINING"]["NUM_HARD_NEGATIVES"]


In [ ]:
en_best_st_model = load_best_model(EN_EFFECTIVE_CONFIG)
en_dev_resource_cache = {}
en_dev_predictions_df, en_dev_metrics = evaluate_on_dev(
    data_df=en_data_dev,
    vocab_df=en_vocab,
    st_model=en_best_st_model,
    cfg=EN_EFFECTIVE_CONFIG,
    resource_cache=en_dev_resource_cache,
)
EN_EFFECTIVE_CONFIG, en_dev_metrics


In [ ]:
en_test_vocab = build_test_inference_vocab(vocab, EN_ENTITIES_FOR_VOCAB_ENRICHMENT, EN_EFFECTIVE_CONFIG, dataset_name="en", lang_value="EN")
en_test_resource_cache = {}
en_test_predictions_df, en_test_predictions_path = predict_on_test(
    data_df=en_data_test,
    vocab_df=en_test_vocab,
    st_model=en_best_st_model,
    cfg=EN_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir("en") / "test_predictions.tsv",
    resource_cache=en_test_resource_cache,
)
en_test_metrics = evaluate_dev_predictions(predictions_df=en_test_predictions_df, data_df=en_data_test)
en_test_metrics


In [ ]:
en_artifact_paths = save_local_artifacts(
    dataset_name="en",
    base_cfg=EN_CONFIG,
    effective_cfg=EN_EFFECTIVE_CONFIG,
    dev_predictions_df=en_dev_predictions_df,
    dev_metrics=en_dev_metrics,
    test_predictions_df=en_test_predictions_df,
    test_metrics=en_test_metrics,
    training_result=en_training_result,
    test_predictions_path=en_test_predictions_path,
)
en_artifact_paths


In [ ]:
en_mlflow_run_id = log_run_to_mlflow(
    dataset_name="en",
    base_cfg=EN_CONFIG,
    effective_cfg=EN_EFFECTIVE_CONFIG,
    dev_metrics=en_dev_metrics,
    test_metrics=en_test_metrics,
    training_result=en_training_result,
    artifact_paths=en_artifact_paths,
)
en_artifact_paths, en_mlflow_run_id


In [ ]:
del en_best_st_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Bilingual Experiment


In [ ]:
BILINGUAL_CONFIG = build_base_experiment_config("andorei/BERGAMOT-multilingual-GAT")

In [ ]:
bilingual_vocab = enrich_vocab_with_oov_train_dev_terms(vocab.copy(), BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT)
bilingual_vocab = prepare_experiment_vocab(
    bilingual_vocab,
    BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT,
    BILINGUAL_CONFIG["INFERENCE"],
)
bilingual_training_result, BILINGUAL_EFFECTIVE_CONFIG = run_training(
    dataset_name="bilingual",
    train_df=bilingual_data_train,
    dev_df=bilingual_data_dev,
    vocab_df=bilingual_vocab,
    cfg=BILINGUAL_CONFIG,
)
len(bilingual_training_result["train_pairs_df"]), BILINGUAL_EFFECTIVE_CONFIG["TRAINING"]["BEST_EPOCH"], BILINGUAL_EFFECTIVE_CONFIG["TRAINING"]["NUM_HARD_NEGATIVES"]


In [ ]:
bilingual_best_st_model = load_best_model(BILINGUAL_EFFECTIVE_CONFIG)
bilingual_dev_resource_cache = {}
bilingual_dev_predictions_df, bilingual_dev_metrics = evaluate_on_dev(
    data_df=bilingual_data_dev,
    vocab_df=bilingual_vocab,
    st_model=bilingual_best_st_model,
    cfg=BILINGUAL_EFFECTIVE_CONFIG,
    resource_cache=bilingual_dev_resource_cache,
)
BILINGUAL_EFFECTIVE_CONFIG, bilingual_dev_metrics


In [ ]:
bilingual_test_vocab = build_test_inference_vocab(vocab, BILINGUAL_ENTITIES_FOR_VOCAB_ENRICHMENT, BILINGUAL_EFFECTIVE_CONFIG, dataset_name="bilingual")
bilingual_test_resource_cache = {}
bilingual_test_predictions_df, bilingual_test_predictions_path = predict_on_test(
    data_df=bilingual_data_test,
    vocab_df=bilingual_test_vocab,
    st_model=bilingual_best_st_model,
    cfg=BILINGUAL_EFFECTIVE_CONFIG,
    output_path=get_artifact_dir("bilingual") / "test_predictions.tsv",
    resource_cache=bilingual_test_resource_cache,
)
bilingual_test_metrics = evaluate_dev_predictions(predictions_df=bilingual_test_predictions_df, data_df=bilingual_data_test)
bilingual_test_metrics


In [ ]:
bilingual_artifact_paths = save_local_artifacts(
    dataset_name="bilingual",
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_EFFECTIVE_CONFIG,
    dev_predictions_df=bilingual_dev_predictions_df,
    dev_metrics=bilingual_dev_metrics,
    test_predictions_df=bilingual_test_predictions_df,
    test_metrics=bilingual_test_metrics,
    training_result=bilingual_training_result,
    test_predictions_path=bilingual_test_predictions_path,
)
bilingual_artifact_paths


In [ ]:
bilingual_mlflow_run_id = log_run_to_mlflow(
    dataset_name="bilingual",
    base_cfg=BILINGUAL_CONFIG,
    effective_cfg=BILINGUAL_EFFECTIVE_CONFIG,
    dev_metrics=bilingual_dev_metrics,
    test_metrics=bilingual_test_metrics,
    training_result=bilingual_training_result,
    artifact_paths=bilingual_artifact_paths,
)
bilingual_artifact_paths, bilingual_mlflow_run_id


In [ ]:
del bilingual_best_st_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
